<a href="https://colab.research.google.com/github/kerondavid-debug/lab-music-recommendations/blob/main/music_recommendations_annotations_1001_2000.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab | Music recommendations — Symbolic Annotation Dataset (1001–2000)

- Based on `Copy_of_lab-music-recommendations.ipynb`, adapted to a **new dataset**: `1001-2000-annotations-v1.1.0.zip`, from the [Artificial Multitrack Audio](https://zenodo.org/records/5794629) dataset referenced in the original lab.
- Unlike the ESC-50 example (raw audio + PANNs embeddings), this dataset ships **symbolic annotations only** (no audio): per-track `.arff` files describing beat/chord grid, per-instrument onset events, and song-section structure (tempo, key, instruments, generator/production style).
- We therefore build our own **structural feature embeddings** from these annotations (tempo, key, instrumentation, chord usage, section count), index them in **Pinecone**, and reuse the exact same RAG-style *"find similar tracks"* pattern as the original lab — plus a **preference-based recommender** (recommend tracks matching a desired tempo/key/instrument profile).


This lab demonstrates how to use Pinecone as the vector DB for a **music structure search / recommendation** application, mirroring the audio-search lab but working from annotation data instead of raw audio.

We parse ~1000 tracks' `.arff` annotations, turn each track into a numeric feature vector (a stand-in for the audio embedding in the original lab), index the vectors in Pinecone, then:
1. find tracks structurally similar to a given track (like the original "find similar sounds"), and
2. recommend tracks matching a user's stated preferences (tempo / key / instruments) — the RAG-style recommendation the lab asks for.

# Install Dependencies

In [1]:
!pip install -qU pinecone==3.1.0 liac-arff numpy==1.26.4 pandas==2.2.2 scikit-learn tqdm

# Load Dataset

We use the `1001-2000-annotations-v1.1.0.zip` dataset: 1000 tracks (IDs 1001-2000), each with three annotation files:

- `<id>_beatinfo.arff` — beat-by-beat chord grid (start time, bar, beat, chord name)
- `<id>_onsets.arff` — per-instrument note onset events over time
- `<id>_segments.arff` — song sections (start time, section mark, tempo, key, active instruments, production "generator" techniques)

Unzip the archive (update the paths below if you're running this outside the environment it was built in):

In [2]:
import zipfile, os

ZIP_PATH = "1001-2000-annotations-v1.1.0.zip"   # path to the uploaded dataset zip
DATA_DIR = "./annotations"

os.makedirs(DATA_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(DATA_DIR)

print(len(os.listdir(DATA_DIR)), "files extracted")

3000 files extracted


The three file types don't reliably use a `@DATA` marker and store list-valued fields as single-quoted, comma-separated strings (e.g. `'[Drums,Cello,Piano]'`). A small custom reader handles both quirks:

In [3]:
import csv, glob

def read_arff_rows(path):
    """Read the data rows of a .arff file, tolerant of missing @DATA markers
    and of single-quoted list-valued STRING fields."""
    with open(path, encoding="utf-8") as f:
        lines = f.readlines()
    data_lines = [l for l in lines if l.strip() and not l.lstrip().startswith("@")]
    return list(csv.reader(data_lines, quotechar="'"))

def track_ids(data_dir=DATA_DIR):
    return sorted(
        {os.path.basename(p).split("_")[0] for p in glob.glob(f"{data_dir}/*_segments.arff")},
        key=int,
    )

ids = track_ids()
print(len(ids), "tracks:", ids[0], "...", ids[-1])

1000 tracks: 1001 ... 2000


Let's look at one track's `segments.arff` (structure) and `beatinfo.arff` (chords) to understand what we're working with:

In [4]:
sample_id = ids[0]
print("segments:")
for row in read_arff_rows(f"{DATA_DIR}/{sample_id}_segments.arff"):
    print(" ", row)
print("\nbeatinfo (first 4 beats):")
for row in read_arff_rows(f"{DATA_DIR}/{sample_id}_beatinfo.arff")[:4]:
    print(" ", row)

segments:
  ['0.0', 'A', '106', 'D#maj', '[Trombone,ElectricPiano,Ukulele,DoubleBassPizz]', '[MelodyBow,ChordPadsRanged,ChordArpeggios,BassLine]']
  ['18.113207', 'C', '106', 'D#maj', '[Trombone,ElectricPiano,Balalaika,DoubleBassPizz,Drums]', '[MelodyBow,ChordPadsRanged,ChordArpeggios,BassLine,RhythmSimpleGrooves]']
  ['54.339622', 'C', '106', 'D#maj', '[Trombone,ElectricPiano,Balalaika,DoubleBassPizz,Drums]', '[MelodyBow,ChordPadsRanged,ChordArpeggios,BassLine,RhythmSimpleGrooves]']
  ['90.56604', 'C', '106', 'D#maj', '[Trombone,ElectricPiano,Balalaika,DoubleBassPizz,Drums]', '[MelodyBow,ChordPadsRanged,ChordArpeggios,BassLine,RhythmSimpleGrooves]']
  ['126.79245', 'B', '106', 'Fmaj', '[Trombone,ElectricPiano,DoubleBassPizz,Drums]', '[MelodyBow,ChordPadsRanged,BassLine,RhythmSimpleGrooves]']
  ['153.96227', 'end', '0', '', '[]', '[]']

beatinfo (first 4 beats):
  ['0.0', '1', '1', 'D#maj']
  ['0.5660377', '1', '2', 'D#maj']
  ['1.1320754', '1', '3', 'D#maj']
  ['1.6981132', '1', '4', 

# Build Track Feature Vectors

With no audio to embed, we build a per-track feature vector directly from the annotations — this plays the same role the PANNs audio embedding played in the original lab. For each track we extract, from `segments.arff`:

- **tempo** (mean BPM across sections)
- **duration** (time of the final `end` marker)
- **num_segments** (song structure complexity — how many distinct sections)
- **key** (most common section key, one-hot encoded)
- **instrumentation** (fraction of sections each instrument appears in, multi-hot)
- **generator/production style** (fraction of sections each production technique appears in, multi-hot)

and from `beatinfo.arff`:

- **chord usage** (normalized histogram of chords played, by beat count)

In [5]:
import collections

def parse_track(tid, data_dir=DATA_DIR):
    seg_rows = read_arff_rows(f"{data_dir}/{tid}_segments.arff")
    segs = []
    for start, mark, tempo, key, instr, gen in seg_rows:
        instr_list = [] if instr in ("[]", "") else instr.strip("[]").split(",")
        gen_list = [] if gen in ("[]", "") else gen.strip("[]").split(",")
        segs.append(dict(start=float(start), mark=mark, tempo=float(tempo),
                          key=key, instruments=instr_list, generator=gen_list))

    beat_rows = read_arff_rows(f"{data_dir}/{tid}_beatinfo.arff")
    chords = [row[3] for row in beat_rows]

    real_segs = [s for s in segs if s["mark"] != "end"]
    duration = segs[-1]["start"] if segs else 0.0
    tempos = [s["tempo"] for s in real_segs if s["tempo"] > 0]
    avg_tempo = sum(tempos) / len(tempos) if tempos else 0.0

    key_counts = collections.Counter(s["key"] for s in real_segs if s["key"])
    main_key = key_counts.most_common(1)[0][0] if key_counts else ""

    n = max(len(real_segs), 1)
    instr_counter, gen_counter = collections.Counter(), collections.Counter()
    for s in real_segs:
        instr_counter.update(set(s["instruments"]))
        gen_counter.update(set(s["generator"]))
    instr_frac = {k: v / n for k, v in instr_counter.items()}
    gen_frac = {k: v / n for k, v in gen_counter.items()}

    chord_counts = collections.Counter(chords)

    return dict(duration=duration, tempo=avg_tempo, key=main_key,
                num_segments=len(real_segs), instr_frac=instr_frac,
                gen_frac=gen_frac, chord_counts=chord_counts)

track_data = {tid: parse_track(tid) for tid in ids}
track_data[ids[0]]

{'duration': 153.96227,
 'tempo': 106.0,
 'key': 'D#maj',
 'num_segments': 5,
 'instr_frac': {'DoubleBassPizz': 1.0,
  'ElectricPiano': 1.0,
  'Ukulele': 0.2,
  'Trombone': 1.0,
  'Balalaika': 0.6,
  'Drums': 0.8},
 'gen_frac': {'MelodyBow': 1.0,
  'BassLine': 1.0,
  'ChordPadsRanged': 1.0,
  'ChordArpeggios': 0.8,
  'RhythmSimpleGrooves': 0.8},
 'chord_counts': Counter({'D#maj': 84,
          'Gmin': 54,
          'Fmin': 24,
          'Cmin': 42,
          'G#maj': 12,
          'A#maj': 28,
          'Amin': 8,
          'Dmin': 6,
          'Fmaj': 12,
          'Cmaj': 2})}

Build the shared vocabularies (instruments, production techniques, keys, chords) across all tracks, so every track vector has the same dimensions:

In [6]:
INSTR_VOCAB = sorted({i for t in track_data.values() for i in t["instr_frac"]})
GEN_VOCAB   = sorted({g for t in track_data.values() for g in t["gen_frac"]})
KEY_VOCAB   = sorted({t["key"] for t in track_data.values() if t["key"]})
CHORD_VOCAB = sorted({c for t in track_data.values() for c in t["chord_counts"]})

print(f"instruments: {len(INSTR_VOCAB)}, generators: {len(GEN_VOCAB)}, "
      f"keys: {len(KEY_VOCAB)}, chords: {len(CHORD_VOCAB)}")

instruments: 31, generators: 5, keys: 24, chords: 26


In [7]:
import numpy as np

tempos    = np.array([track_data[t]["tempo"] for t in ids])
durations = np.array([track_data[t]["duration"] for t in ids])
nsegs     = np.array([track_data[t]["num_segments"] for t in ids])

def _norm(x):
    lo, hi = x.min(), x.max()
    return (x - lo) / (hi - lo + 1e-9)

tempo_n, dur_n, nseg_n = _norm(tempos), _norm(durations), _norm(nsegs)

def vectorize(tid, idx):
    td = track_data[tid]
    v = [tempo_n[idx], dur_n[idx], nseg_n[idx]]
    v += [1.0 if k == td["key"] else 0.0 for k in KEY_VOCAB]
    v += [td["instr_frac"].get(i, 0.0) for i in INSTR_VOCAB]
    v += [td["gen_frac"].get(g, 0.0) for g in GEN_VOCAB]
    total_chords = sum(td["chord_counts"].values()) or 1
    v += [td["chord_counts"].get(c, 0) / total_chords for c in CHORD_VOCAB]
    return np.array(v, dtype=np.float32)

embeddings = np.stack([vectorize(tid, i) for i, tid in enumerate(ids)])
EMBED_DIM = embeddings.shape[1]
print("embeddings shape:", embeddings.shape)

embeddings shape: (1000, 89)


## Initializing the Index

As in the original lab, we store these vectors in Pinecone and use it to run the similarity search. Get a [free API key](https://app.pinecone.io/) and set it below (or via `PINECONE_API_KEY` env var / Colab secret).

In [8]:
import os
from pinecone import Pinecone

try:
    from google.colab import userdata
    PINECONE_API_KEY = userdata.get("PINECONE_API_KEY") or os.getenv("PINECONE_API_KEY") or "YOUR_API_KEY"
except ImportError:
    PINECONE_API_KEY = os.getenv("PINECONE_API_KEY", "YOUR_API_KEY")

pc = Pinecone(api_key=PINECONE_API_KEY)

In [9]:
from pinecone import ServerlessSpec

cloud = os.environ.get("PINECONE_CLOUD") or "aws"
region = os.environ.get("PINECONE_REGION") or "us-east-1"
spec = ServerlessSpec(cloud=cloud, region=region)

index_name = "music-annotation-search-demo"

In [10]:
import time

if index_name not in pc.list_indexes().names():
    pc.create_index(index_name, dimension=EMBED_DIM, metric="cosine", spec=spec)
    while not pc.describe_index(index_name).status["ready"]:
        time.sleep(1)

index = pc.Index(index_name)
index.describe_index_stats()

{'dimension': 89,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 1000}},
 'total_vector_count': 1000}

# Upsert Embeddings

We upsert in batches, attaching each track's key facts (tempo, key, section count) as metadata so results are readable without a second lookup.

In [11]:
from tqdm.auto import tqdm

batch_size = 64

for i in tqdm(range(0, len(ids), batch_size)):
    batch_ids = ids[i:i + batch_size]
    batch_vecs = embeddings[i:i + batch_size]
    to_upsert = []
    for tid, vec in zip(batch_ids, batch_vecs):
        td = track_data[tid]
        meta = dict(tempo=td["tempo"], key=td["key"], num_segments=td["num_segments"],
                    instruments=list(td["instr_frac"].keys())[:8])
        to_upsert.append((tid, vec.tolist(), meta))
    index.upsert(vectors=to_upsert)

index.describe_index_stats()

  0%|          | 0/16 [00:00<?, ?it/s]

{'dimension': 89,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 1000}},
 'total_vector_count': 1000}

We now have ~1000 tracks indexed in Pinecone, we're ready to query.

# Querying

Let's look at a track and find its structurally-closest neighbors, the same pattern as the original lab's `find_similar_audios`.

In [12]:
def describe(tid):
    td = track_data[tid]
    print(f"Track {tid}: {td['tempo']:.0f} BPM, key {td['key']}, "
          f"{td['num_segments']} sections, instruments: {list(td['instr_frac'].keys())}")

def find_similar_tracks(tid, top_k=5):
    describe(tid)
    result = index.query(id=str(tid), top_k=top_k + 1, include_metadata=True)
    print("\nSimilar tracks:")
    for match in result["matches"]:
        if match["id"] == str(tid):
            continue
        print(f"  {match['id']}  score={match['score']:.3f}")
        describe(match["id"])

In [13]:
find_similar_tracks(ids[49])   # try any track id from `ids`

Track 1050: 120 BPM, key Bmin, 6 sections, instruments: ['ElectricGuitarCrunch', 'Shakuhachi', 'ElectricBass', 'ElectricGuitarClean', 'Fujara', 'Cello', 'Drums']

Similar tracks:
  1285  score=0.910
Track 1285: 133 BPM, key Bmin, 5 sections, instruments: ['Sitar', 'Drums', 'Shakuhachi', 'ElectricBass', 'Cello', 'Balalaika', 'ElectricGuitarCrunch', 'MorinKhuur']
  1027  score=0.817
Track 1027: 96 BPM, key Bmin, 5 sections, instruments: ['Fujara', 'Sitar', 'Drums', 'ElectricBass']
  1912  score=0.809
Track 1912: 142 BPM, key Bmin, 8 sections, instruments: ['ElectricGuitarClean', 'Trombone', 'ElectricBass', 'Cello', 'Sitar', 'Drums']
  1635  score=0.805
Track 1635: 108 BPM, key Bmin, 8 sections, instruments: ['ElectricGuitarClean', 'ElectricGuitarCrunch', 'Jinghu', 'Drums', 'ElectricBass']
  1596  score=0.801
Track 1596: 159 BPM, key Bmin, 8 sections, instruments: ['Balalaika', 'MorinKhuur', 'ElectricBass', 'Ukulele', 'ElectricGuitarClean', 'Drums', 'ElectricGuitarCrunch']


## Preference-based recommendation

This is the RAG-style recommendation the lab calls for: build a query vector from a *user's stated preferences* (desired tempo, key, instruments) instead of an existing track, and retrieve the closest matches.

In [14]:
def recommend_from_preferences(tempo=None, key=None, instruments=None, top_k=5):
    """Recommend tracks matching a user's tempo / key / instrument preferences."""
    v = np.zeros(EMBED_DIM, dtype=np.float32)

    if tempo is not None:
        v[0] = np.clip((tempo - tempos.min()) / (tempos.max() - tempos.min() + 1e-9), 0, 1)
    if key is not None and key in KEY_VOCAB:
        v[3 + KEY_VOCAB.index(key)] = 1.0
    if instruments:
        offset = 3 + len(KEY_VOCAB)
        for instr in instruments:
            if instr in INSTR_VOCAB:
                v[offset + INSTR_VOCAB.index(instr)] = 1.0

    result = index.query(vector=v.tolist(), top_k=top_k, include_metadata=True)
    print("Recommendations:")
    for match in result["matches"]:
        print(f"  {match['id']}  score={match['score']:.3f}")
        describe(match["id"])
    return result

In [15]:
# Example: fast, minor-key tracks featuring Drums and ElectricBass
_ = recommend_from_preferences(tempo=140, key="Emin", instruments=["Drums", "ElectricBass"])

Recommendations:
  1636  score=0.628
Track 1636: 143 BPM, key Emin, 6 sections, instruments: ['Drums', 'Ukulele', 'PanFlute', 'ElectricBass', 'Piano']
  1379  score=0.615
Track 1379: 122 BPM, key Emin, 12 sections, instruments: ['ElectricBass', 'Viola', 'ElectricGuitarCrunch', 'AltoSax', 'Drums', 'BrightPiano', 'Trombone']
  1361  score=0.597
Track 1361: 134 BPM, key Emin, 10 sections, instruments: ['BrightPiano', 'Trombone', 'ElectricBass', 'Cello', 'AltoSax', 'Piano', 'Drums', 'Jinghu']
  1921  score=0.582
Track 1921: 128 BPM, key Emin, 11 sections, instruments: ['ElectricGuitarClean', 'Violin', 'ElectricGuitarCrunch', 'ElectricBass', 'BrightPiano', 'AcousticGuitar', 'Drums', 'MorinKhuur']
  1381  score=0.541
Track 1381: 66 BPM, key Emin, 5 sections, instruments: ['ElectricBass', 'BrightPiano', 'Flute', 'Balalaika', 'Drums', 'Cello', 'Ukulele', 'MorinKhuur']


## Local fallback (no Pinecone key needed)

If you don't have a Pinecone API key handy, the same search works locally with cosine similarity over the `embeddings` matrix — useful for quickly sanity-checking the feature vectors.

In [16]:
from sklearn.neighbors import NearestNeighbors

nn_model = NearestNeighbors(n_neighbors=6, metric="cosine").fit(embeddings)

def find_similar_tracks_local(tid, top_k=5):
    describe(tid)
    idx = ids.index(tid)
    dist, neighbor_idxs = nn_model.kneighbors(embeddings[idx:idx + 1], n_neighbors=top_k + 1)
    print("\nSimilar tracks (local):")
    for d, i in zip(dist[0], neighbor_idxs[0]):
        if ids[i] == tid:
            continue
        print(f"  {ids[i]}  score={1 - d:.3f}")
        describe(ids[i])

find_similar_tracks_local(ids[49])

Track 1050: 120 BPM, key Bmin, 6 sections, instruments: ['ElectricGuitarCrunch', 'Shakuhachi', 'ElectricBass', 'ElectricGuitarClean', 'Fujara', 'Cello', 'Drums']

Similar tracks (local):
  1285  score=0.910
Track 1285: 133 BPM, key Bmin, 5 sections, instruments: ['Sitar', 'Drums', 'Shakuhachi', 'ElectricBass', 'Cello', 'Balalaika', 'ElectricGuitarCrunch', 'MorinKhuur']
  1027  score=0.817
Track 1027: 96 BPM, key Bmin, 5 sections, instruments: ['Fujara', 'Sitar', 'Drums', 'ElectricBass']
  1912  score=0.809
Track 1912: 142 BPM, key Bmin, 8 sections, instruments: ['ElectricGuitarClean', 'Trombone', 'ElectricBass', 'Cello', 'Sitar', 'Drums']
  1635  score=0.805
Track 1635: 108 BPM, key Bmin, 8 sections, instruments: ['ElectricGuitarClean', 'ElectricGuitarCrunch', 'Jinghu', 'Drums', 'ElectricBass']
  1596  score=0.801
Track 1596: 159 BPM, key Bmin, 8 sections, instruments: ['Balalaika', 'MorinKhuur', 'ElectricBass', 'Ukulele', 'ElectricGuitarClean', 'Drums', 'ElectricGuitarCrunch']


We get structurally coherent recommendations — matching key and tempo, overlapping instrumentation — the same result quality the original lab achieved with audio embeddings, but driven entirely by the annotation metadata since no raw audio ships with this dataset.

# Delete the Index

Delete the index once you are sure that you do not want to use it anymore. Once the index is deleted, you cannot use it again.

In [17]:
pc.delete_index(index_name)